# Schema dump — the ~40 Drive pkls behind the 32 Colab-era sweep cells

Follow-up to `coverage_audit.csv`: the first audit only recognized canonical key
names, and the oldest caches (phase4/5 gpqa+math500, phase9 qa, phase10 RAG,
phase13) predate the standardized schema. This dump records each pkl's REAL
structure — container type, entry keys, value types/shapes, and 200-char string
previews (which shows whether RAG prompts/passages were saved).

**CPU runtime, ~5-15 min** (small files, targeted list — not a full Drive walk).
Output: `MyDrive/hallucination_detection/schema_dump.json` — drop it into the repo
root like you did with coverage_audit.csv (or leave it on Drive and tell Claude).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import glob, json, os, pickle, sys
import numpy as np

ROOT = '/content/drive/MyDrive'
OUT  = '/content/drive/MyDrive/hallucination_detection/schema_dump.json'

# Only the dirs that back the 32 sweep-pool cells (+ the two verify caches).
TARGETS = [
    'epr_spectral_phase1/spectral_cache.pkl',
    'epr_spectral_phase3/*/phase3_cache.pkl',
    'epr_spectral_phase4/*/inference_cache.pkl',
    'epr_spectral_phase5/*/inference_cache.pkl',
    'epr_spectral_phase6/*/inference_cache.pkl',
    'epr_spectral_gpqa_72b/*/inference_cache.pkl',
    'epr_spectral_gsm8k_vs_lapei/*/inference_cache.pkl',
    'spectral_phase9_cache/*.pkl',
    'hallucination_detection/cache/phase10_main/data_*.pkl',
    'hallucination_detection/cache/phase10_main/raw/*.pkl',
    'hallucination_detection/cache/phase10_main/features/qwen7b__hotpotqa.pkl',
    'hallucination_detection/cache/mathcomp_phase13/*/*.pkl',
    'hallucination_detection/cache/phase15_temperature/math500_qwen7b_T1.0_run0.pkl',
    'hallucination_detection/phase_gsm8k_verify/inference_*.pkl',
    'hallucination_detection/phase_spilled_verify/inference_*.pkl',
]

PREVIEW = 200

def describe(v, depth=0):
    """Short structural descriptor; recurses one level into dicts/lists."""
    if v is None:
        return 'None'
    if isinstance(v, dict):
        d = {'__type': 'dict', '__len': len(v)}
        if depth < 2:
            for k in list(v)[:25]:
                d[repr(k)] = describe(v[k], depth + 1)
        return d
    if isinstance(v, (list, tuple)):
        d = {'__type': type(v).__name__, '__len': len(v)}
        if len(v) and depth < 2:
            d['__elem0'] = describe(v[0], depth + 1)
        return d
    if isinstance(v, np.ndarray):
        return 'ndarray shape=%s dtype=%s' % (v.shape, v.dtype)
    if isinstance(v, str):
        s = v[:PREVIEW].replace('\n', '\\n')
        return 'str[%d] %r' % (len(v), s)
    if isinstance(v, (int, float, bool, np.integer, np.floating, np.bool_)):
        return '%s=%r' % (type(v).__name__, v)
    return type(v).__name__

records = []
paths = []
for pat in TARGETS:
    paths += sorted(glob.glob(os.path.join(ROOT, pat)))
print('%d target pkls' % len(paths))

for i, p in enumerate(paths):
    rel = os.path.relpath(p, ROOT)
    print('(%d/%d) %s' % (i + 1, len(paths), rel), flush=True)
    rec = {'path': rel, 'size_mb': round(os.path.getsize(p) / 1e6, 1)}
    try:
        with open(p, 'rb') as f:
            obj = pickle.load(f)
    except Exception as e:
        rec['error'] = '%s: %s' % (type(e).__name__, e)
        records.append(rec)
        continue
    rec['top_type'] = type(obj).__name__
    if isinstance(obj, dict):
        rec['top_len'] = len(obj)
        rec['top_keys_sample'] = [repr(k) for k in list(obj)[:8]]
        # dump the structure of the first TWO entries in full (depth-2)
        rec['entries'] = {repr(k): describe(obj[k]) for k in list(obj)[:2]}
    elif isinstance(obj, (list, tuple)):
        rec['top_len'] = len(obj)
        rec['entries'] = {'[0]': describe(obj[0]) if len(obj) else 'empty',
                          '[1]': describe(obj[1]) if len(obj) > 1 else '-'}
    else:
        rec['entries'] = describe(obj)
    records.append(rec)

with open(OUT, 'w') as f:
    json.dump(records, f, indent=1)
print('\nwrote', OUT)
